# Predict Parent Motivation — Per-Subdomain LOSO

Does prediction accuracy vary by subdomain, and does subdomain-specific modeling improve
on the universal baseline?

**CV scheme**: Leave-One-Subdomain-Out (LOSO) — 16 folds; sparse subdomains (n<5) noted.

**Three tiers**:
1. Universal LOSO: structured features, train on all other subdomains
2. +Population history: training-fold motivation-class proportions (8 features)
3. +Per-subdomain history: test subdomain's other highlights' motivation distribution
   (oracle; domain-level fallback for sparse subdomains)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

DATA_DIR   = Path('../../data-exports/20260412_183830')
OUTPUT_DIR = DATA_DIR / 'highlight_analysis_output'
OUT_DIR    = OUTPUT_DIR / 'motivation_classifier_output'
OUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE  = 42
SPARSE_THRESH = 5

## Load data

In [ ]:
df_sel = pd.read_csv(OUTPUT_DIR / 'df_sel.csv')

r5 = pd.read_csv(DATA_DIR / 'R5_highlights_coded', sep='\t')
r5.columns = r5.columns.str.strip()
r5 = r5.rename(columns={
    'highlight_id':      'selection_id',
    'Parent Motivation': 'r5_motivation',
    'Model Strategy':    'r5_strategy',
})
r5['r5_motivation'] = r5['r5_motivation'].replace({
    '': None, 'null': None,
    'Response Could Evoke Strong Emotion': 'Response Could Evoke Strong Emotions',
    'Parents Trust of Model Capabilities': None,
})
r5 = r5[r5['r5_motivation'].notna()]

df = df_sel.merge(r5[['selection_id', 'r5_motivation']], on='selection_id', how='inner')
df = df.drop(columns=['parent_motivation'], errors='ignore').rename(
    columns={'r5_motivation': 'parent_motivation'}
).reset_index(drop=True)

print(f'Shape: {df.shape}')
print('Subdomains:')
print(df['subdomain'].value_counts().sort_values().to_string())
sparse = df['subdomain'].value_counts()
sparse = sparse[sparse < SPARSE_THRESH].index.tolist()
print(f'\nSparse (n<{SPARSE_THRESH}): {sparse}')

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(df['parent_motivation'])
CLASSES   = le.classes_
N_CLASSES = len(CLASSES)

print(f'{N_CLASSES} motivation classes:')
for i, (cls, n) in enumerate(zip(CLASSES, np.bincount(y))):
    print(f'  [{i}] {cls}: {n}')


def build_features(df_in: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df_in.index)
    out['strategy_is_null'] = df_in['model_strategy'].isna().astype(int)
    out = pd.concat([out,
        pd.get_dummies(df_in['model_strategy'].fillna('none'), prefix='strat'),
    ], axis=1)
    for col in ['domain', 'age_band', 'sensitivity_level', 'relationship_frame',
                'space_type', 'trait', 'trait_level']:
        out = pd.concat([out,
            pd.get_dummies(df_in[col].fillna('unknown'), prefix=col)
        ], axis=1)
    out['breakdown_expected'] = (df_in['breakdown_expected'] == 'yes').astype(int)
    if 'source' in df_in.columns:
        out = pd.concat([out,
            pd.get_dummies(df_in['source'].fillna('unknown'), prefix='source')
        ], axis=1)
    for col in ['parent_gender', 'parent_age_group', 'parent_education', 'parent_ethnicity',
                'area_of_residency', 'child_has_ai_use', 'parent_llm_monitoring_level']:
        out = pd.concat([out,
            pd.get_dummies(df_in[col].fillna('unknown'), prefix=col)
        ], axis=1)
    out['genai_regular_user'] = (df_in['genai_familiarity'] == 'regular_user').astype(int)
    freq_map = {'daily': 2, 'weekly': 1, 'monthly_or_less': 0}
    out['genai_usage_freq'] = df_in['genai_usage_frequency'].map(freq_map).fillna(0).astype(int)
    out['parent_internet_use_frequency'] = pd.to_numeric(
        df_in['parent_internet_use_frequency'], errors='coerce').fillna(0)
    out['is_only_child'] = (df_in['is_only_child'].astype(str).str.lower() == 'yes').astype(int)
    for s in 'ABCD':
        out[f'parenting_{s}'] = df_in['parenting_style'].fillna('').str.contains(s).astype(int)
    return out.astype(float)


X_struct = build_features(df)
print(f'\nStructured features: {X_struct.shape[1]}')

In [ ]:
def top2_acc(y_true: np.ndarray, y_proba: np.ndarray) -> float:
    top2 = np.argsort(y_proba, axis=1)[:, -2:]
    return float(np.mean([y_true[i] in top2[i] for i in range(len(y_true))]))


LOSO_MODELS = {
    'Majority Baseline':   DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=300, max_depth=8,
                                                   random_state=RANDOM_STATE),
}

## Tier 1 — Universal LOSO

In [ ]:
def run_loso_structured(df, X, y, models: dict) -> pd.DataFrame:
    rows = []
    for sd in df['subdomain'].unique():
        test_mask = (df['subdomain'] == sd).values
        y_te      = y[test_mask]
        n         = test_mask.sum()

        if n < 2 or len(np.unique(y_te)) < 2:
            for name in models:
                rows.append({'subdomain': sd, 'n': n, 'model': name,
                             'f1_weighted': np.nan, 'top2_acc': np.nan,
                             'note': 'insufficient'})
            continue

        y_tr = y[~test_mask]
        for name, model in models.items():
            model.fit(X[~test_mask], y_tr)
            y_pred = model.predict(X[test_mask])
            f1w    = f1_score(y_te, y_pred, average='weighted', zero_division=0)
            t2a = np.nan
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(X[test_mask])
                if proba.shape[1] == N_CLASSES:
                    t2a = top2_acc(y_te, proba)
            rows.append({
                'subdomain':   sd, 'n': n, 'model': name,
                'f1_weighted': round(f1w, 3),
                'top2_acc':    round(t2a, 3) if not np.isnan(t2a) else np.nan,
                'note':        'sparse' if n < SPARSE_THRESH else '',
            })
    return pd.DataFrame(rows)


loso_universal = run_loso_structured(df, X_struct.values, y, LOSO_MODELS)
print('Tier 1 — Universal LOSO (mean across subdomains):')
print(loso_universal.groupby('model')[['f1_weighted', 'top2_acc']].mean().round(3).to_string())
loso_universal.to_csv(OUT_DIR / 'motivation_per_subdomain_universal.csv', index=False)

## Tier 2 — +Population history

In [ ]:
def build_pop_history(y_train: np.ndarray, n_rows: int, n_classes: int) -> np.ndarray:
    counts = np.bincount(y_train, minlength=n_classes)
    props  = counts / counts.sum()
    return np.tile(props, (n_rows, 1))


def run_loso_with_pop_history(df, X_struct, y, models: dict) -> pd.DataFrame:
    rows = []
    for sd in df['subdomain'].unique():
        test_mask = (df['subdomain'] == sd).values
        y_te      = y[test_mask]
        n         = test_mask.sum()

        if n < 2 or len(np.unique(y_te)) < 2:
            for name in models:
                rows.append({'subdomain': sd, 'n': n, 'model': name,
                             'f1_weighted': np.nan, 'top2_acc': np.nan,
                             'note': 'insufficient'})
            continue

        y_tr  = y[~test_mask]
        pop_tr = build_pop_history(y_tr, (~test_mask).sum(), N_CLASSES)
        pop_te = build_pop_history(y_tr,   test_mask.sum(),  N_CLASSES)
        X_tr   = np.hstack([X_struct[~test_mask], pop_tr])
        X_te   = np.hstack([X_struct[test_mask],  pop_te])

        for name, model in models.items():
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            f1w    = f1_score(y_te, y_pred, average='weighted', zero_division=0)
            t2a = np.nan
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(X_te)
                if proba.shape[1] == N_CLASSES:
                    t2a = top2_acc(y_te, proba)
            rows.append({
                'subdomain':   sd, 'n': n, 'model': name,
                'f1_weighted': round(f1w, 3),
                'top2_acc':    round(t2a, 3) if not np.isnan(t2a) else np.nan,
                'note':        'sparse' if n < SPARSE_THRESH else '',
            })
    return pd.DataFrame(rows)


loso_history = run_loso_with_pop_history(df, X_struct.values, y, LOSO_MODELS)
print('Tier 2 — +Population history (mean across subdomains):')
print(loso_history.groupby('model')[['f1_weighted', 'top2_acc']].mean().round(3).to_string())
loso_history.to_csv(OUT_DIR / 'motivation_per_subdomain_loso.csv', index=False)

## Tier 3 — +Per-subdomain history (oracle; domain fallback for sparse)

For each test row of subdomain S, compute the motivation-class distribution from
S's other test rows.  Domain-level fallback when the subdomain pool has < 5 rows.

In [ ]:
def _make_fallback(df_pool: pd.DataFrame, y_pool: np.ndarray,
                   subdomain: str, n_classes: int) -> np.ndarray:
    """Domain-level fallback, then global, for sparse subdomains."""
    domain_mask = df_pool['domain'] == df_pool.loc[
        df_pool['subdomain'] == subdomain, 'domain'
    ].iloc[0] if (df_pool['subdomain'] == subdomain).any() else np.zeros(len(df_pool), bool)
    pool_mask = domain_mask if domain_mask.sum() > 0 else np.ones(len(df_pool), bool)
    y_p = y_pool[pool_mask]
    counts = np.bincount(y_p, minlength=n_classes)
    return counts / counts.sum()


def build_sd_history(df_rows: pd.DataFrame, y_rows: np.ndarray,
                     fallback: np.ndarray, n_classes: int) -> np.ndarray:
    result = []
    for i in range(len(df_rows)):
        sd     = df_rows.iloc[i]['subdomain']
        others = [j for j in range(len(df_rows))
                  if j != i and df_rows.iloc[j]['subdomain'] == sd]
        if not others:
            result.append(fallback)
        else:
            counts = np.bincount(y_rows[others], minlength=n_classes)
            result.append(counts / counts.sum())
    return np.array(result)


def run_loso_with_sd_history(df, X_struct, y, models: dict) -> pd.DataFrame:
    rows = []
    for sd in df['subdomain'].unique():
        test_mask = (df['subdomain'] == sd).values
        df_tr     = df[~test_mask].reset_index(drop=True)
        df_te     = df[test_mask].reset_index(drop=True)
        y_tr, y_te = y[~test_mask], y[test_mask]
        n          = test_mask.sum()

        if n < 2 or len(np.unique(y_te)) < 2:
            for name in models:
                rows.append({'subdomain': sd, 'n': n, 'model': name,
                             'f1_weighted': np.nan, 'top2_acc': np.nan,
                             'note': 'insufficient'})
            continue

        pop_fb = np.bincount(y_tr, minlength=N_CLASSES) / len(y_tr)

        sd_tr = build_sd_history(df_tr, y_tr, pop_fb, N_CLASSES)
        sd_te = build_sd_history(df_te, y_te, pop_fb, N_CLASSES)

        X_tr = np.hstack([X_struct[~test_mask], sd_tr])
        X_te = np.hstack([X_struct[test_mask],  sd_te])

        for name, model in models.items():
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            f1w    = f1_score(y_te, y_pred, average='weighted', zero_division=0)
            t2a = np.nan
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(X_te)
                if proba.shape[1] == N_CLASSES:
                    t2a = top2_acc(y_te, proba)
            rows.append({
                'subdomain':   sd, 'n': n, 'model': name,
                'f1_weighted': round(f1w, 3),
                'top2_acc':    round(t2a, 3) if not np.isnan(t2a) else np.nan,
                'note':        'sparse' if n < SPARSE_THRESH else '',
            })
    return pd.DataFrame(rows)


loso_sd = run_loso_with_sd_history(df, X_struct.values, y, LOSO_MODELS)
print('Tier 3 — +Per-subdomain history (mean across subdomains):')
print(loso_sd.groupby('model')[['f1_weighted', 'top2_acc']].mean().round(3).to_string())
loso_sd.to_csv(OUT_DIR / 'motivation_per_subdomain_loso_per_subdomain.csv', index=False)

## Summary table

In [ ]:
summary_rows = []
for tier, loso_df in [
    ('Universal',      loso_universal),
    ('+Pop history',   loso_history),
    ('+Per-subdomain', loso_sd),
]:
    for model in ['Majority Baseline', 'Logistic Regression', 'Random Forest']:
        sub = loso_df[loso_df['model'] == model]
        summary_rows.append({
            'tier':        tier,
            'model':       model,
            'f1_weighted': round(sub['f1_weighted'].mean(), 3),
            'top2_acc':    round(sub['top2_acc'].mean(), 3),
        })

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))
summary.to_csv(OUT_DIR / 'motivation_per_subdomain_summary.csv', index=False)

## Per-subdomain F1 heatmap

In [ ]:
# Pivot for RF: subdomains × tiers
def pivot_rf(df_loso, tier_label):
    sub = df_loso[df_loso['model'] == 'Random Forest'][['subdomain', 'n', 'f1_weighted']]
    return sub.rename(columns={'f1_weighted': tier_label})

hm = (
    pivot_rf(loso_universal, 'Universal')
    .merge(pivot_rf(loso_history,  '+Pop history').drop(columns='n'),  on='subdomain')
    .merge(pivot_rf(loso_sd, '+Per-subdomain').drop(columns='n'), on='subdomain')
    .sort_values('n')
    .set_index('subdomain')
)

fig, ax = plt.subplots(figsize=(9, max(5, len(hm) * 0.45)))
sns.heatmap(
    hm[['Universal', '+Pop history', '+Per-subdomain']].astype(float),
    annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1.0,
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Weighted F1'},
)
ax.set_title('Per-subdomain motivation prediction — Random Forest LOSO (sorted by n)')
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig(OUT_DIR / 'motivation_per_subdomain_heatmap.png', dpi=100)
plt.show()